In [20]:
raw_data = [
    (1, "Laptop", "Electronics", 1200.0, 1, "Completed"),
    (2, "Desk Chair", "Furniture", 250.0, 2, "Completed"),
    (3, "Wireless Mouse", "Electronics", 35.0, 4, "Completed"),
    (4, "Monitor", "Electronics", None, 1, "Completed"),
    (5, "Standing Desk", "Furniture", 550.0, 1, "Completed"),
    (6, "Webcam", "Electronics", 85.0, -2, "Completed"),
    (7, "Notebook Pack", "Stationery", 12.0, 10, "Cancelled"),
    (8, "Desk Lamp", "Furniture", 45.0, 3, "Completed"),
    (9, "USB-C Hub", "Electronics", 60.0, 2, "Completed"),
    (10, "Pen Set", "Stationery", 15.0, 5, "Completed"),
]

bad_columns = [
    "Order ID",
    "Product Name",
    "Category",
    "Unit Price ($)",
    "Qty",
    "Order_Status",
]

### PANDAS EAGER EVALUATION

In [26]:
import pandas as pd

# Step 1: Load data into memory immediately
pdf = pd.DataFrame(raw_data, columns=bad_columns)


# Step 2: Rename columns immediately
pdf = pdf.rename(
    columns={
        "Unit Price ($)": "unit_price",
        "Qty": "quantity",
        "Order_Status": "status",
    }
)


# Step 3: Filter rows immediately (CPU already doing work)
pdf = pdf[
    (pdf["status"] == "Completed")
    & (pdf["unit_price"].notna())
    & (pdf["quantity"] > 0)
].copy()



# Step 4: Calculate revenue immediately (More compute burned)
pdf["revenue"] = pdf["unit_price"] * pdf["quantity"]


# Step 5: Category Typo -> Crashes AFTER Steps 1-4 already ran and burned compute
result = pdf.groupby("typo_category")["revenue"].sum()

KeyError: 'typo_category'

### SPARK LAZY EVALUATION

In [28]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySpark_Workshop")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

# Step 1: Create DataFrame (Lazy — nothing executed yet)
df = spark.createDataFrame(raw_data, schema=bad_columns)

# Step 2: Rename columns (Lazy — Catalyst Analyzer checks column names)
df_renamed = (
    df.withColumnRenamed("Unit Price ($)", "unit_price")
    .withColumnRenamed("Qty", "quantity")
    .withColumnRenamed("Order_Status", "status")
)

# Step 3: Filter rows (Lazy — zero workers touched)
df_filtered = df_renamed.filter(
    (F.col("status") == "Completed")
    & (F.col("unit_price").isNotNull())
    & (F.col("quantity") > 0)
)

# Step 4: Calculate revenue (Lazy — still zero compute used)
df_revenue = df_filtered.withColumn("revenue", F.col("unit_price") * F.col("quantity"))

# Step 5: Category Typo -> Catalyst Analyzer catches this immediately on the Driver!
df_grouped = df_revenue.groupBy("Category").agg(
    F.sum("revenue").alias("total_revenue")
)

# Step 6: Action 
df_grouped.show()

[Stage 0:>                                                          (0 + 2) / 2]

+-----------+-------------+
|   Category|total_revenue|
+-----------+-------------+
|Electronics|       1460.0|
|  Furniture|       1185.0|
| Stationery|         75.0|
+-----------+-------------+



In [29]:
spark.stop()